# Nike Sales Data Cleaning Project

## Objective
Demonstrate professional-level data cleaning by transforming a deliberately messy Nike sales dataset into a clean, analysis-ready dataset.

**Tech Stack:** Python, pandas, numpy, Jupyter Notebook

### Cleaning principles
- Inspect before changing data.
- Document every cleaning decision.
- Treat logically invalid values separately from statistical outliers.
- Avoid inappropriate imputation for fields where a plausible value cannot be inferred.
- Validate the final dataset and export it as a new CSV.


## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)


## 2. Load the Uncleaned Dataset

In [2]:
# Keep Nike_Sales_Uncleaned.csv in the same folder as this notebook.
file_path = "Nike_Sales_Uncleaned.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
display(df.head())


Dataset shape: (2500, 13)


,Order_ID,Gender_Category,Product_Line,Product_Name,Size,Units_Sold,MRP,Discount_Applied,Revenue,Order_Date,Sales_Channel,Region,Profit
0,2000,Kids,Training,SuperRep Go,M,NaN,NaN,0.47,0.0,2024-03-09,Online,bengaluru,-770.45
1,2001,Women,Soccer,Tiempo Legend,M,3.0,4957.93,NaN,0.0,2024-07-09,Retail,Hyd,-112.53
2,2002,Women,Soccer,Premier III,M,4.0,NaN,NaN,0.0,NaN,Retail,Mumbai,3337.34
3,2003,Kids,Lifestyle,Blazer Mid,L,NaN,9673.57,NaN,0.0,04-10-2024,Online,Pune,3376.85
4,2004,Kids,Running,React Infinity,XL,NaN,NaN,NaN,0.0,2024/09/12,Retail,Delhi,187.89


## 3. Initial Data Inspection

In [3]:
print("Shape:", df.shape)
print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values per column:")
display(df.isna().sum().to_frame("null_count"))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nNumeric summary:")
display(df.describe())


Shape: (2500, 13)

Data types:


,dtype
Order_ID,int64
Gender_Category,object
Product_Line,object
Product_Name,object
Size,object
Units_Sold,float64
MRP,float64
Discount_Applied,float64
Revenue,float64
Order_Date,object



Missing values per column:


,null_count
Order_ID,0
Gender_Category,0
Product_Line,0
Product_Name,0
Size,510
Units_Sold,1235
MRP,1254
Discount_Applied,1668
Revenue,0
Order_Date,616



Duplicate rows: 0

Numeric summary:


,Order_ID,Units_Sold,MRP,Discount_Applied,Revenue,Profit
count,2500.000000,1265.000000,1246.000000,832.000000,2500.000000,2500.000000
mean,3194.352400,1.482213,6039.863395,0.631022,274.873716,1376.012848
std,749.972638,1.696947,2315.746826,0.365500,2023.773550,1478.671013
min,2000.000000,-1.000000,2006.630000,0.000000,-7561.590000,-1199.450000
25%,2534.750000,0.000000,4038.902500,0.320000,0.000000,93.532500
50%,3192.500000,2.000000,6110.030000,0.615000,0.000000,1371.385000
75%,3841.250000,3.000000,8022.187500,0.962500,0.000000,2660.645000
max,4499.000000,4.000000,9996.220000,1.250000,37169.350000,3999.210000


## 4. Data Quality Report

The quality report checks:
1. Null values per column.
2. Duplicate rows.
3. Data type problems.
4. Logical/range anomalies.
5. Inconsistent categorical values.

The raw dataset contains missing values in `Size`, `Units_Sold`, `MRP`, `Discount_Applied`, and `Order_Date`. It also contains inconsistent region labels and logically invalid numeric values such as negative units sold and discounts greater than 100%.


In [4]:
quality_report = pd.DataFrame({
    "Null_Count": df.isna().sum(),
    "Unique_Values": df.nunique(dropna=True),
    "Data_Type": df.dtypes.astype(str)
})

print("Duplicate rows:", df.duplicated().sum())
display(quality_report)

print("Units_Sold unique values:", sorted(df["Units_Sold"].dropna().unique()))
print("Discount values above 1:", (df["Discount_Applied"] > 1).sum())
print("Negative Revenue values:", (df["Revenue"] < 0).sum())
print("Region values:", sorted(df["Region"].unique()))


Duplicate rows: 0


,Null_Count,Unique_Values,Data_Type
Order_ID,0,2386,int64
Gender_Category,0,3,object
Product_Line,0,5,object
Product_Name,0,20,object
Size,510,10,object
Units_Sold,1235,6,float64
MRP,1254,1244,float64
Discount_Applied,1668,126,float64
Revenue,0,167,float64
Order_Date,616,1008,object


Units_Sold unique values: [np.float64(-1.0), np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
Discount values above 1: 180
Negative Revenue values: 53
Region values: ['Bangalore', 'Delhi', 'Hyd', 'Hyderabad', 'Kolkata', 'Mumbai', 'Pune', 'bengaluru', 'hyderbad']


## 5. Missing Data Handling Strategy

### `Order_Date` — Row deletion
Order date is an essential transaction field. There is no reliable way to infer an exact date from the other columns. Forward-filling would incorrectly assign one customer's transaction date to another transaction. Therefore, rows with missing dates are removed.

### `Units_Sold` — Median imputation
Units sold is numeric and contains an invalid `-1` value. Negative units are treated as missing first. Median is used because it is robust to skew and unusual values.

### `MRP` — Median imputation
MRP is numeric and represents a monetary value. Median imputation avoids the influence of unusually high product prices.

### `Discount_Applied` — Median imputation
Discount is numeric and should be between 0 and 1. Values above 1 are treated as invalid/missing first, then the median is used.

### `Size` — Mode imputation
Size is categorical. The most frequent category is a suitable simple replacement when the missing value cannot be inferred from other columns.

### Other columns
The remaining categorical and numeric fields have no missing values in the source dataset, so no imputation is required.


In [5]:
# Work on a copy so the original data remains unchanged.
cleaned = df.copy()

# 1. Standardise Region labels
region_map = {
    "bengaluru": "Bangalore",
    "hyderbad": "Hyderabad",
    "Hyd": "Hyderabad"
}
cleaned["Region"] = cleaned["Region"].replace(region_map)

# 2. Convert logically invalid values to missing before imputation
cleaned.loc[cleaned["Units_Sold"] < 0, "Units_Sold"] = np.nan
cleaned.loc[
    (cleaned["Discount_Applied"] < 0) | (cleaned["Discount_Applied"] > 1),
    "Discount_Applied"
] = np.nan

# 3. Parse mixed date formats
def parse_mixed_date(value):
    if pd.isna(value):
        return pd.NaT

    value = str(value)

    for fmt in ("%d-%m-%Y", "%Y/%m/%d", "%Y-%m-%d"):
        parsed = pd.to_datetime(value, format=fmt, errors="coerce")
        if pd.notna(parsed):
            return parsed

    return pd.NaT

cleaned["Order_Date"] = cleaned["Order_Date"].map(parse_mixed_date)

# 4. Remove rows with missing essential dates
rows_before_date_removal = len(cleaned)
cleaned = cleaned.dropna(subset=["Order_Date"]).copy()
rows_removed_for_date = rows_before_date_removal - len(cleaned)

# 5. Impute missing values
cleaned["Size"] = cleaned["Size"].fillna(cleaned["Size"].mode()[0])
cleaned["Units_Sold"] = cleaned["Units_Sold"].fillna(cleaned["Units_Sold"].median())
cleaned["MRP"] = cleaned["MRP"].fillna(cleaned["MRP"].median())
cleaned["Discount_Applied"] = cleaned["Discount_Applied"].fillna(
    cleaned["Discount_Applied"].median()
)

print("Rows removed because Order_Date was missing:", rows_removed_for_date)


Rows removed because Order_Date was missing: 616


## 6. Duplicate Removal

Exact duplicate rows are identified using `duplicated()`. Duplicate transactions would inflate sales and profit calculations, so exact duplicates are removed.

**Note:** The source dataset contains no exact duplicate rows, so the number removed is expected to be zero.


In [6]:
duplicates_before = cleaned.duplicated().sum()
cleaned = cleaned.drop_duplicates().copy()

print("Duplicate rows identified and removed:", duplicates_before)


Duplicate rows identified and removed: 0


## 7. Standardisation

### Region
Different spellings and abbreviations are normalised:
- `bengaluru` → `Bangalore`
- `Hyd` → `Hyderabad`
- `hyderbad` → `Hyderabad`

### Dates
The source contains multiple formats such as:
- `DD-MM-YYYY`
- `YYYY/MM/DD`
- `YYYY-MM-DD`

All valid dates are converted to pandas `datetime64[ns]`.


In [7]:
print("Standardised regions:")
display(sorted(cleaned["Region"].unique()))

print("\nDate range:")
print(cleaned["Order_Date"].min(), "to", cleaned["Order_Date"].max())


Standardised regions:


['Bangalore', 'Delhi', 'Hyderabad', 'Kolkata', 'Mumbai', 'Pune']


Date range:
2023-07-26 00:00:00 to 2025-07-22 00:00:00


## 8. Outlier Detection Using the IQR Method

The IQR method is applied to numeric variables before median imputation, after obvious logical errors have been converted to missing values.

**Important distinction:** a logical anomaly is not automatically a statistical outlier. For example, `Units_Sold = -1` is invalid even though it may fall inside an IQR boundary. Therefore, logical validation and statistical outlier detection are both performed.

For `Revenue`, the raw data contains a very large number of zero values, making the standard IQR rule unsuitable for meaningful business outlier detection. Revenue is therefore validated through business logic and recalculated from cleaned transaction inputs instead.


In [8]:
outlier_results = []

outlier_data = df.copy()

# Remove logical invalids before statistical testing
outlier_data.loc[outlier_data["Units_Sold"] < 0, "Units_Sold"] = np.nan
outlier_data.loc[
    (outlier_data["Discount_Applied"] < 0) |
    (outlier_data["Discount_Applied"] > 1),
    "Discount_Applied"
] = np.nan

for col in ["Units_Sold", "MRP", "Discount_Applied", "Profit"]:
    series = outlier_data[col].dropna()

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    count = ((series < lower) | (series > upper)).sum()

    outlier_results.append({
        "Column": col,
        "Q1": q1,
        "Q3": q3,
        "Lower_Bound": lower,
        "Upper_Bound": upper,
        "Outlier_Count": count
    })

outlier_report = pd.DataFrame(outlier_results)
display(outlier_report)


,Column,Q1,Q3,Lower_Bound,Upper_Bound,Outlier_Count
0,Units_Sold,1.0000,3.0000,-2.00000,6.00000,0
1,MRP,4038.9025,8022.1875,-1936.02500,13997.11500,0
2,Discount_Applied,0.2400,0.7500,-0.52500,1.51500,0
3,Profit,93.5325,2660.6450,-3757.13625,6511.31375,0


### Outlier Decision

No statistically extreme observations were removed from `Units_Sold`, `MRP`, `Discount_Applied`, or `Profit` using the IQR rule.

However, the following **logical anomalies** were corrected:
- Negative `Units_Sold` values were converted to missing and median-imputed.
- `Discount_Applied > 1` was treated as invalid and median-imputed.
- Negative/invalid revenue was not blindly capped; revenue was recalculated from the cleaned MRP, units sold, and discount values.

This preserves valid high-value transactions instead of deleting legitimate business observations.


## 9. Revenue Consistency Correction

Revenue is a derived business measure. After cleaning its component fields, it is recalculated using:

`Revenue = MRP × Units_Sold × (1 − Discount_Applied)`

This prevents impossible negative revenue caused by invalid units or discounts and ensures consistency between the transaction inputs and the revenue column.


In [9]:
cleaned["Revenue"] = (
    cleaned["MRP"]
    * cleaned["Units_Sold"]
    * (1 - cleaned["Discount_Applied"])
).round(2)

print("Negative revenue after recalculation:", (cleaned["Revenue"] < 0).sum())
display(cleaned[["MRP", "Units_Sold", "Discount_Applied", "Revenue"]].head())


Negative revenue after recalculation:

 0


,MRP,Units_Sold,Discount_Applied,Revenue
0,6100.23,2.0,0.47,6466.24
1,4957.93,3.0,0.48,7734.37
3,9673.57,2.0,0.48,10060.51
4,6100.23,2.0,0.48,6344.24
6,6819.78,2.0,0.48,7092.57


## 10. Data Type Correction

Expected final data types:
- `Order_ID` → string
- Categorical/text columns → string
- `Units_Sold`, `MRP`, `Discount_Applied`, `Revenue`, `Profit` → float
- `Order_Date` → datetime


In [10]:
cleaned["Order_ID"] = cleaned["Order_ID"].astype("string")
cleaned["Gender_Category"] = cleaned["Gender_Category"].astype("string")
cleaned["Product_Line"] = cleaned["Product_Line"].astype("string")
cleaned["Product_Name"] = cleaned["Product_Name"].astype("string")
cleaned["Size"] = cleaned["Size"].astype("string")
cleaned["Units_Sold"] = cleaned["Units_Sold"].astype("float64")
cleaned["MRP"] = cleaned["MRP"].astype("float64")
cleaned["Discount_Applied"] = cleaned["Discount_Applied"].astype("float64")
cleaned["Revenue"] = cleaned["Revenue"].astype("float64")
cleaned["Order_Date"] = pd.to_datetime(cleaned["Order_Date"])
cleaned["Sales_Channel"] = cleaned["Sales_Channel"].astype("string")
cleaned["Region"] = cleaned["Region"].astype("string")
cleaned["Profit"] = cleaned["Profit"].astype("float64")

display(cleaned.dtypes.to_frame("Final_Dtype"))


,Final_Dtype
Order_ID,string[python]
Gender_Category,string[python]
Product_Line,string[python]
Product_Name,string[python]
Size,string[python]
Units_Sold,float64
MRP,float64
Discount_Applied,float64
Revenue,float64
Order_Date,datetime64[ns]


## 11. Before vs After Summary

The summary compares row count, total missing values, duplicate rows, and expected data-type accuracy before and after cleaning.


In [11]:
expected_dtypes = {
    "Order_ID": "string",
    "Gender_Category": "string",
    "Product_Line": "string",
    "Product_Name": "string",
    "Size": "string",
    "Units_Sold": "float64",
    "MRP": "float64",
    "Discount_Applied": "float64",
    "Revenue": "float64",
    "Order_Date": "datetime64[ns]",
    "Sales_Channel": "string",
    "Region": "string",
    "Profit": "float64"
}

before_dtype_accuracy = (
    sum(str(df[col].dtype) == dtype for col, dtype in expected_dtypes.items())
    / len(expected_dtypes) * 100
)

after_dtype_accuracy = (
    sum(str(cleaned[col].dtype) == dtype for col, dtype in expected_dtypes.items())
    / len(expected_dtypes) * 100
)

before_after = pd.DataFrame({
    "Metric": [
        "Row count",
        "Total null values",
        "Duplicate rows",
        "Dtype accuracy"
    ],
    "Before Cleaning": [
        len(df),
        int(df.isna().sum().sum()),
        int(df.duplicated().sum()),
        f"{before_dtype_accuracy:.1f}%"
    ],
    "After Cleaning": [
        len(cleaned),
        int(cleaned.isna().sum().sum()),
        int(cleaned.duplicated().sum()),
        f"{after_dtype_accuracy:.1f}%"
    ]
})

display(before_after)


,Metric,Before Cleaning,After Cleaning
0,Row count,2500,1884
1,Total null values,5283,0
2,Duplicate rows,0,0
3,Dtype accuracy,38.5%,100.0%


## 12. Final Validation

In [12]:
print("Final shape:", cleaned.shape)
print("Total missing values:", cleaned.isna().sum().sum())
print("Duplicate rows:", cleaned.duplicated().sum())
print("Negative Units_Sold:", (cleaned["Units_Sold"] < 0).sum())
print("Invalid Discount values:", ((cleaned["Discount_Applied"] < 0) | (cleaned["Discount_Applied"] > 1)).sum())
print("Negative Revenue:", (cleaned["Revenue"] < 0).sum())

assert cleaned.isna().sum().sum() == 0
assert cleaned.duplicated().sum() == 0
assert (cleaned["Units_Sold"] >= 0).all()
assert ((cleaned["Discount_Applied"] >= 0) & (cleaned["Discount_Applied"] <= 1)).all()
assert (cleaned["Revenue"] >= 0).all()

print("\nAll validation checks passed.")


Final shape: (1884, 13)
Total missing values: 0
Duplicate rows: 0
Negative Units_Sold: 0
Invalid Discount values: 0
Negative Revenue: 0

All validation checks passed.


## 13. Export the Cleaned Dataset

In [13]:
output_file = "Nike_Sales_Cleaned.csv"
cleaned.to_csv(output_file, index=False)

print(f"Cleaned dataset saved successfully as: {output_file}")


Cleaned dataset saved successfully as: Nike_Sales_Cleaned.csv


## Conclusion

The Nike sales dataset was systematically transformed from a messy source into an analysis-ready dataset.

Key improvements:
- Missing values were handled using column-appropriate strategies.
- Missing transaction dates were removed because they could not be reliably inferred.
- Exact duplicate rows were checked and removed.
- Region names were standardised.
- Mixed date formats were converted to a consistent datetime format.
- Invalid units and discounts were corrected before imputation.
- Statistical outliers were evaluated using the IQR method.
- Revenue was recalculated from cleaned transaction inputs.
- Data types were corrected and validated.
- A before-vs-after quality summary was produced.
- The final dataset was exported as `Nike_Sales_Cleaned.csv`.

The cleaned file is now suitable for downstream analysis and visualisation.
